In [ ]:
pip install brevitas

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

import brevitas.nn as qnn
from brevitas.quant import Int8WeightPerTensorFixedPoint
from brevitas.quant import Int8ActPerTensorFixedPoint

# ==========================================================
# CONFIGURAÇÃO
# ==========================================================

In [ ]:
BIT_WIDTH = 8  # <-- ALTERE AQUI (8, 6, 4, etc.)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 5
BATCH_SIZE = 128
PRECISION_N=32

# ==========================================================
# DATASET MNIST
# ==========================================================

In [ ]:

transform = transforms.Compose([
    transforms.ToTensor(),
])

trainset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=BATCH_SIZE, shuffle=True)

testset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=BATCH_SIZE, shuffle=False)

# ==========================================================
# MODELO LENET QUANTIZADO
# ==========================================================

In [ ]:
from torch.nn import Module
import torch.nn.functional as F

import brevitas.nn as qnn
from brevitas.quant import Int32Bias, Int8ActPerTensorFloat, Int8ActPerTensorFixedPoint
from brevitas.core.scaling import PowerOfTwoIntScaling


class QuantWeightActBiasLeNet(Module):
    def __init__(self):
        super(QuantWeightActBiasLeNet, self).__init__()
        self.quant_inp = qnn.QuantIdentity(bit_width=BIT_WIDTH, return_quant_tensor=True)
        self.conv1 = qnn.QuantConv2d(1, 6, 5,padding=2, bias=False, weight_bit_width=BIT_WIDTH, output_quant=Int8ActPerTensorFixedPoint, return_quant_tensor=True)
        #self.relu1 = qnn.QuantReLU(bit_width=BIT_WIDTH, output_quant=Int8ActPerTensorFixedPoint, return_quant_tensor=True)
        self.conv2 = qnn.QuantConv2d(6, 16, 5, bias=False, weight_bit_width=BIT_WIDTH, output_quant=Int8ActPerTensorFixedPoint, return_quant_tensor=True)
        #self.relu2 = qnn.QuantReLU(bit_width=BIT_WIDTH, output_quant=Int8ActPerTensorFixedPoint, return_quant_tensor=True)
        self.fc1   = qnn.QuantLinear(16*5*5, 120, bias=False, output_quant=Int8ActPerTensorFixedPoint, return_quant_tensor=True)
        #self.relu3 = qnn.QuantReLU(bit_width=BIT_WIDTH, output_quant=Int8ActPerTensorFixedPoint, return_quant_tensor=True)
        self.fc2   = qnn.QuantLinear(120, 84, bias=False, output_quant=Int8ActPerTensorFixedPoint, return_quant_tensor=True)
        #self.relu4 = qnn.QuantReLU(bit_width=BIT_WIDTH, output_quant=Int8ActPerTensorFixedPoint, return_quant_tensor=True)
        self.fc3   = qnn.QuantLinear(84, 10, bias=False, output_quant=Int8ActPerTensorFixedPoint, return_quant_tensor=True)

    def forward(self, x):
        out = self.quant_inp(x)
        #out = self.relu1(self.conv1(out))
        out = F.relu(self.conv1(out))
        out = F.max_pool2d(out, 2)
        #out = self.relu2(self.conv2(out))
        out = F.relu(self.conv2(out))
        out = F.max_pool2d(out, 2)
        out = out.reshape(out.shape[0], -1)
        #out = self.relu3(self.fc1(out))
        out = F.relu(self.fc1(out))
        #out = self.relu4(self.fc2(out))
        out = F.relu(self.fc2(out))
        out = self.fc3(out)
        return out

# ==========================================================
# TREINAMENTO
# ==========================================================

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_loss = total_loss / len(loader)

    return avg_loss, accuracy

In [ ]:
model = QuantWeightActBiasLeNet().to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_loss = running_loss / len(trainloader)
    test_loss, test_acc = evaluate(model, testloader)
    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Test  Loss: {test_loss:.4f}")
    print(f"Test  Acc : {test_acc:.2f}%")
    print("-"*40)

print("Treinamento finalizado.")


/usr/local/lib/python3.12/dist-packages/torch/_tensor.py:1630: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at /pytorch/c10/core/TensorImpl.h:1971.)
  return super().rename(names)


Epoch 1
Train Loss: 0.3623
Test  Loss: 0.1169
Test  Acc : 96.21%
----------------------------------------
Epoch 2
Train Loss: 0.0936
Test  Loss: 0.0592
Test  Acc : 98.07%
----------------------------------------
Epoch 3
Train Loss: 0.0667
Test  Loss: 0.0543
Test  Acc : 98.17%
----------------------------------------
Epoch 4
Train Loss: 0.0546
Test  Loss: 0.0427
Test  Acc : 98.54%
----------------------------------------
Epoch 5
Train Loss: 0.0433
Test  Loss: 0.0416
Test  Acc : 98.42%
----------------------------------------
Treinamento finalizado.


==========================================================
#VERIFICAÇÃO
==========================================================


In [ ]:
def get_layer_output_int8_from_intermediate(layer, input_tensor):
    """
    Recebe diretamente o tensor que é a entrada REAL da camada.
    Não executa o modelo inteiro.
    """

    layer.eval()

    with torch.no_grad():

        # Executa apenas a camada
        x = layer(input_tensor)

        output_scale = layer.output_quant.scale().item()

        out_int8 = torch.round(
            x / output_scale
        ).clamp(-128, 127).to(torch.int8)

    return out_int8[0]


In [ ]:
import torch
import torch.nn.functional as F

def validate_conv_layer(model, layer, input_tensor, previous_layer=None, k=5, max_lines=800):

    model.cpu()
    model.eval()

    patches = []

    # =========================================================
    # 1) INPUT INT E SCALE CORRETO
    # =========================================================
    with torch.no_grad():

        if previous_layer is None:
            # Primeira camada
            q_input = model.quant_inp(input_tensor)
            input_scale = model.quant_inp.act_quant.scale().item()
        else:
            # Camadas internas
            q_input = input_tensor
            input_scale = previous_layer.output_quant.scale().item()

    input_int = torch.round(
        q_input / input_scale
    ).clamp(-128, 127).to(torch.int8)

    # Remove batch e canal único (MNIST)
    #input_int = input_int[0, 0]

    # =========================================================
    # 2) PESOS
    # =========================================================
    weight_int = layer.quant_weight().int()
    weight_scale = layer.weight_quant.scale().detach()
    output_scale = layer.output_quant.scale().item()

    num_filters = weight_int.shape[0]
    input_int = input_int[0]

    # =========================================================
    # 3) MULTIPLICADORES
    # =========================================================
    M_list = []
    Mint_list = []

    print("\n=========== MULTIPLICADORES POR FILTRO ===========")

    for f in range(num_filters):

        if weight_scale.numel() == 1:
            w_scale = weight_scale.item()
        else:
            w_scale = weight_scale[f].item()

        M = (input_scale * w_scale) / output_scale
        Mint = int(round(M * (1 << PRECISION_N)))

        M_list.append(M)
        Mint_list.append(Mint)

        print(f"\nFiltro {f}:")
        print("Input scale       :", input_scale)
        print("Weight scale      :", w_scale)
        print("Output scale      :", output_scale)
        print("Multiplicador M   :", M)
        print("Multiplicador Mint:", Mint)

    # =========================================================
    # 4) OUTPUT REAL DO MODELO
    # =========================================================
    model_out_int8 = get_layer_output_int8_from_intermediate(layer, input_tensor)
    # =========================================================
    # 5) CONVOLUÇÃO MANUAL
    # =========================================================


    padding = layer.padding[0] if isinstance(layer.padding, tuple) else layer.padding

    if padding > 0:
        input_int = F.pad(
            input_int,
            (padding, padding, padding, padding),
            mode="constant",
            value=0
        )
    print(input_int.shape)
    C, H, W = input_int.shape
    results = []
    input_id = 0

    for h in range(H - k + 1):
        for w in range(W - k + 1):

            patches.append(input_int[:, h:h+k, w:w+k])

            acc_list = []
            manual_int8_list = []
            model_int8_list = []

            for f in range(num_filters):

                acc_int32 = 0

                for c in range(weight_int.shape[1]):  # in_channels

                    kernel_int = weight_int[f, c]
                    patch_c = input_int[c, h:h+k, w:w+k]

                    acc_int32 += (patch_c.int() * kernel_int.int()).sum()

                # float version
                manual_int8 = torch.round(
                    acc_int32.float() * M_list[f]
                ).clamp(-128, 127).to(torch.int8)

                model_int8 = model_out_int8[f, h, w]

                acc_list.append(acc_int32.item())
                manual_int8_list.append(manual_int8.item())
                model_int8_list.append(model_int8.item())

            results.append((input_id,
                            model_int8_list,
                            manual_int8_list,
                            acc_list))

            input_id += 1

    # =========================================================
    # 6) PRINT FINAL
    # =========================================================
    print("\nINPUT_ID | INT8(Model) | INT8(ManualFloat) | INT8(ManualInt) | ACC32")
    print("-" * 120)

    for r in results[:max_lines]:

        idx, model_vals, manual_vals, acc_vals = r

        manual2_vals = []

        for f in range(len(acc_vals)):

            acc = acc_vals[f]
            Mint = Mint_list[f]

            # Requantização inteira pura (igual hardware)
            manual2 = ((acc * Mint) + (1 << (PRECISION_N - 1))) >> PRECISION_N
            manual2 = max(-128, min(127, manual2))

            manual2_vals.append(manual2)

        model_str   = " ".join([f"{v:4d}" for v in model_vals])
        manual_str  = " ".join([f"{v:4d}" for v in manual_vals])
        manual2_str = " ".join([f"{v:4d}" for v in manual2_vals])
        acc_str     = " ".join([f"{v:6d}" for v in acc_vals])

        print(f"{idx:8d} | {model_str} | {manual_str} | {manual2_str} | {acc_str}")

    return patches


In [ ]:
validate_conv_layer
dataiter = iter(testloader)
image, label = next(dataiter)
patches_preconv1=validate_conv_layer(model, model.conv1, image, previous_layer=None, k=5, max_lines=800)


=========== MULTIPLICADORES POR FILTRO ===========

Filtro 0:
Input scale       : 0.007936418056488037
Weight scale      : 0.0034135570749640465
Output scale      : 0.03125
Multiplicador M   : 0.0008669253122111087
Multiplicador Mint: 3723416

Filtro 1:
Input scale       : 0.007936418056488037
Weight scale      : 0.0034135570749640465
Output scale      : 0.03125
Multiplicador M   : 0.0008669253122111087
Multiplicador Mint: 3723416

Filtro 2:
Input scale       : 0.007936418056488037
Weight scale      : 0.0034135570749640465
Output scale      : 0.03125
Multiplicador M   : 0.0008669253122111087
Multiplicador Mint: 3723416

Filtro 3:
Input scale       : 0.007936418056488037
Weight scale      : 0.0034135570749640465
Output scale      : 0.03125
Multiplicador M   : 0.0008669253122111087
Multiplicador Mint: 3723416

Filtro 4:
Input scale       : 0.007936418056488037
Weight scale      : 0.0034135570749640465
Output scale      : 0.03125
Multiplicador M   : 0.0008669253122111087
Multiplicador Mi

In [ ]:
with torch.no_grad():
    x = model.quant_inp(image)
    x = F.relu(model.conv1(x))
    x = F.max_pool2d(x, 2)
    image2=x
patches_preconv2=validate_conv_layer(model, model.conv2, image2, previous_layer=model.conv1, k=5, max_lines=800)


=========== MULTIPLICADORES POR FILTRO ===========

Filtro 0:
Input scale       : 0.03125
Weight scale      : 0.003224111394956708
Output scale      : 0.0625
Multiplicador M   : 0.001612055697478354
Multiplicador Mint: 6923726

Filtro 1:
Input scale       : 0.03125
Weight scale      : 0.003224111394956708
Output scale      : 0.0625
Multiplicador M   : 0.001612055697478354
Multiplicador Mint: 6923726

Filtro 2:
Input scale       : 0.03125
Weight scale      : 0.003224111394956708
Output scale      : 0.0625
Multiplicador M   : 0.001612055697478354
Multiplicador Mint: 6923726

Filtro 3:
Input scale       : 0.03125
Weight scale      : 0.003224111394956708
Output scale      : 0.0625
Multiplicador M   : 0.001612055697478354
Multiplicador Mint: 6923726

Filtro 4:
Input scale       : 0.03125
Weight scale      : 0.003224111394956708
Output scale      : 0.0625
Multiplicador M   : 0.001612055697478354
Multiplicador Mint: 6923726

Filtro 5:
Input scale       : 0.03125
Weight scale      : 0.0032241

============================

#INPUT WINDOW

============================


In [ ]:
for patch in patches_preconv1:
  flat=patch.flatten()
  string = ''
  for f in flat:
    string+=' '+str(f.item())
  print(string)

 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

============================

#EXPORTAR PARA USAR NO HARDWARE

============================


============================

#MOSTRANDO OS PESOS QUANTIZADOS DE CADA CAMADA

============================

======================================

#MOSTRANDO IS INPUTS QUANTIZADOS

======================================

In [ ]:
with torch.no_grad():
    q_input = model.quant_inp(image)
    input_scale = model.quant_inp.act_quant.scale().item()
    input_int = torch.round(q_input / input_scale).clamp(-128, 127).to(torch.int8)
    B, C, H, W = input_int.shape
    input_id = 0
    for h in range(H):
        for w in range(W):
            val = input_int[0, 0, h, w].item()
            #print(f"{input_id:4d} -> {val}")
            print(f"{val}")
            input_id += 1


0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
42
91
79
75
30
18
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
110
126
126
126
126
119
98
98
98
98
98
98
98
98
84
26
0
0
0
0
0
0
0
0
0
0
0
0
33
56
36
56
81
112
126
111
126
126
126
124
113
126
126
69
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
8
33
7
33
33
33
29
10
117
126
52
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
41
125
103
9
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
11
115
126
41
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
64
126
118
22
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
29
123
126
31
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
66
126
92
2
0
0
0
0
0
0
0
0
0
0
0
0
0
0


In [ ]:
import torch
import torch.nn as nn

def print_model_weights_and_mint(model, input_tensor):

    print("\n==================== EXPORTANDO MODELO ====================")

    with torch.no_grad():

        # =========================================================
        # 1) PRIMEIRO SCALE = scale do quant_inp
        # =========================================================
        x = model.quant_inp(input_tensor)
        current_input_scale = model.quant_inp.act_quant.scale().item()

        # Lista manual na ordem do forward
        layers_in_order = [
            model.conv1,
            model.conv2,
            model.fc1,
            model.fc2,
            model.fc3
        ]

        for layer in layers_in_order:

            print(f"\n==================== {layer.__class__.__name__} ====================")

            weight_int = layer.quant_weight().int().cpu()
            weight_scale = layer.weight_quant.scale().detach().cpu()
            output_scale = layer.output_quant.scale().item()

            # -----------------------------------------------------
            # CONVOLUTIONAL
            # -----------------------------------------------------
            if len(weight_int.shape) == 4:

                out_ch = weight_int.shape[0]

                for f in range(out_ch):

                    # PESOS
                    filtro = weight_int[f].reshape(-1)
                    print(" ".join(str(w) for w in filtro.tolist()))

                    # SCALE DO FILTRO
                    if weight_scale.numel() == 1:
                        w_scale = weight_scale.item()
                    else:
                        w_scale = weight_scale[f].item()

                    # MULTIPLICADOR CORRETO
                    M = (current_input_scale * w_scale) / output_scale
                    Mint = int(round(M * (1 << PRECISION_N)))

                    print(Mint)

            # -----------------------------------------------------
            # LINEAR
            # -----------------------------------------------------
            elif len(weight_int.shape) == 2:

                out_f = weight_int.shape[0]

                for n in range(out_f):

                    neuron = weight_int[n]
                    print(" ".join(str(w) for w in neuron.tolist()))

                    if weight_scale.numel() == 1:
                        w_scale = weight_scale.item()
                    else:
                        w_scale = weight_scale[n].item()

                    M = (current_input_scale * w_scale) / output_scale
                    Mint = int(round(M * (1 << PRECISION_N)))

                    print(Mint)

            # =====================================================
            # ATUALIZA SCALE PARA PRÓXIMA CAMADA
            # =====================================================
            current_input_scale = output_scale


In [ ]:
image
print_model_weights_and_mint(model, image)


==================== EXPORTANDO MODELO ====================

==================== QuantConv2d ====================
27 70 80 34 90 100 2 15 28 81 -14 72 1 -4 16 -76 -64 20 13 -46 -28 -99 -103 -94 -56
3723416
-8 -48 -122 -98 -87 -22 -90 -68 -3 -11 40 4 63 43 -10 54 83 10 -6 40 41 77 46 86 74
3723416
-12 -6 36 46 -33 34 -31 -20 68 -15 16 62 53 -23 18 45 1 39 -3 5 -1 58 48 69 44
3723416
-106 -25 56 28 76 -57 28 87 106 24 71 -19 -59 -59 -44 69 2 -52 -101 55 26 -72 40 -7 -23
3723416
-12 27 12 76 42 -58 74 81 99 29 -6 -9 81 24 81 4 -33 -7 7 92 -19 25 14 -19 25
3723416
100 62 3 -40 -31 85 7 -24 -72 -76 6 92 -30 -127 -18 41 43 -72 -85 -28 49 -3 -39 -70 56
3723416

==================== QuantConv2d ====================
-1 -32 -29 28 49 -32 -61 -50 47 72 -35 -68 -70 -70 -46 63 40 -7 31 18 24 23 8 26 40 -26 -100 -127 -77 0 23 6 -37 -88 -48 91 89 42 75 66 9 44 36 43 52 -25 -28 -8 6 -21 -6 -68 -92 -4 -16 38 -8 -66 -25 -28 33 0 -7 29 -9 23 -2 17 -12 12 -11 20 -24 -23 -11 21 -23 -52 -1 24 24 -42 -53 4

============================

#MOSTRANDO AS SAÍDAS DE CADA CAMADA

============================


In [ ]:
import torch

def print_layer_output(model, layer, input_tensor, layer_name="Layer", first_layer=False):

    model.eval()

    with torch.no_grad():

        # # Passa pelo quantizador de entrada se existir
        # if hasattr(model, "quant_inp"):
        #     x = model.quant_inp(input_tensor)
        # else:
        #     x = input_tensor

        if (first_layer):
          x = model.quant_inp(input_tensor)
        else:
          x = input_tensor

        # Forward até a camada desejada
        output = layer(x)

        # Se a camada tiver quantização de saída
        if hasattr(layer, "output_quant"):
            output_scale = layer.output_quant.scale().item()
            output_int = torch.round(
                output / output_scale
            ).clamp(-128,127).to(torch.int8)
        else:
            output_int = output

    output_int = output_int[0]  # remove batch

    print(f"\n==============================")
    print(f"Camada: {layer_name}")
    print(f"==============================\n")

    # ============================================
    # CASO 1: CAMADA CONVOLUCIONAL
    # ============================================
    if output_int.ndim == 3:

        C, H, W = output_int.shape
        num_positions = H * W

        print("ID:", end=" ")
        for c in range(C):
            print(f"Filtro {c:3d}", end="   ")
        print("\n")

        idx = 0
        for h in range(H):
            for w in range(W):

                print(f"{idx:4d}:", end=" ")

                for c in range(C):
                    val = output_int[c, h, w].item()
                    print(f"{val:6d}", end="   ")

                print()
                idx += 1

    # ============================================
    # CASO 2: CAMADA LINEAR / MLP
    # ============================================
    elif output_int.ndim == 1:

        N = output_int.shape[0]

        print("ID:", end=" ")
        for i in range(N):
            print(f"Neuronio {i:3d}", end="   ")
        print("\n")

        print("0:", end=" ")
        for i in range(N):
            val = output_int[i].item()
            print(f"{val:6d}", end="   ")
        print()

    else:
        print("Formato de saída não suportado:", output_int.shape)


In [ ]:
print_layer_output(model, model.conv1, image, layer_name="Conv1", first_layer=True)


Camada: Conv1

ID: Filtro   0   Filtro   1   Filtro   2   Filtro   3   Filtro   4   Filtro   5   

   0:      0        0        0        0        0        0   
   1:      0        0        0        0        0        0   
   2:      0        0        0        0        0        0   
   3:      0        0        0        0        0        0   
   4:      0        0        0        0        0        0   
   5:      0        0        0        0        0        0   
   6:      0        0        0        0        0        0   
   7:      0        0        0        0        0        0   
   8:      0        0        0        0        0        0   
   9:      0        0        0        0        0        0   
  10:      0        0        0        0        0        0   
  11:      0        0        0        0        0        0   
  12:      0        0        0        0        0        0   
  13:      0        0        0        0        0        0   
  14:      0        0        0        0       

In [ ]:
output_conv2=F.max_pool2d(F.relu(model.conv1(model.quant_inp(image))),2)
print_layer_output(model, model.conv2, output_conv2, layer_name="Conv2")


Camada: Conv2

ID: Filtro   0   Filtro   1   Filtro   2   Filtro   3   Filtro   4   Filtro   5   Filtro   6   Filtro   7   Filtro   8   Filtro   9   Filtro  10   Filtro  11   Filtro  12   Filtro  13   Filtro  14   Filtro  15   

   0:     21       -2       -7        0       -7      -31       18        0        2       21        5        2      -16        3      -16       10   
   1:     24        4      -15       -3       -5      -28       22       -7       -1       25       18        0      -19        0      -17       12   
   2:     30        3      -27       -9       -6      -19       22      -12       -1       18       32       -3      -14       -3       -7       16   
   3:     32       -3      -32      -14      -11      -20       24      -13       -2       14       39       -6       -6       -4        4       23   
   4:     29       -8      -29      -16      -13      -29       23      -12       -6       12       36       -7        1       -6       10       27   
   5:     23   

In [ ]:
output_fc1=F.max_pool2d(F.relu(model.conv2(F.max_pool2d(F.relu(model.conv1(model.quant_inp(image))),2))), 2)
output_fc1=output_fc1.reshape(output_fc1.shape[0], -1)
print_layer_output(model, model.fc1, output_fc1, layer_name="Fc1")


Camada: Fc1

ID: Neuronio   0   Neuronio   1   Neuronio   2   Neuronio   3   Neuronio   4   Neuronio   5   Neuronio   6   Neuronio   7   Neuronio   8   Neuronio   9   Neuronio  10   Neuronio  11   Neuronio  12   Neuronio  13   Neuronio  14   Neuronio  15   Neuronio  16   Neuronio  17   Neuronio  18   Neuronio  19   Neuronio  20   Neuronio  21   Neuronio  22   Neuronio  23   Neuronio  24   Neuronio  25   Neuronio  26   Neuronio  27   Neuronio  28   Neuronio  29   Neuronio  30   Neuronio  31   Neuronio  32   Neuronio  33   Neuronio  34   Neuronio  35   Neuronio  36   Neuronio  37   Neuronio  38   Neuronio  39   Neuronio  40   Neuronio  41   Neuronio  42   Neuronio  43   Neuronio  44   Neuronio  45   Neuronio  46   Neuronio  47   Neuronio  48   Neuronio  49   Neuronio  50   Neuronio  51   Neuronio  52   Neuronio  53   Neuronio  54   Neuronio  55   Neuronio  56   Neuronio  57   Neuronio  58   Neuronio  59   Neuronio  60   Neuronio  61   Neuronio  62   Neuronio  63   Neuronio  64   Neuroni

In [ ]:
output_fc2=F.max_pool2d(F.relu(model.conv2(F.max_pool2d(F.relu(model.conv1(model.quant_inp(image))),2))), 2)
output_fc2=output_fc2.reshape(output_fc2.shape[0], -1)
output_fc2=F.relu(model.fc1(output_fc2))
print_layer_output(model, model.fc2, output_fc2, layer_name="Fc2")


Camada: Fc2

ID: Neuronio   0   Neuronio   1   Neuronio   2   Neuronio   3   Neuronio   4   Neuronio   5   Neuronio   6   Neuronio   7   Neuronio   8   Neuronio   9   Neuronio  10   Neuronio  11   Neuronio  12   Neuronio  13   Neuronio  14   Neuronio  15   Neuronio  16   Neuronio  17   Neuronio  18   Neuronio  19   Neuronio  20   Neuronio  21   Neuronio  22   Neuronio  23   Neuronio  24   Neuronio  25   Neuronio  26   Neuronio  27   Neuronio  28   Neuronio  29   Neuronio  30   Neuronio  31   Neuronio  32   Neuronio  33   Neuronio  34   Neuronio  35   Neuronio  36   Neuronio  37   Neuronio  38   Neuronio  39   Neuronio  40   Neuronio  41   Neuronio  42   Neuronio  43   Neuronio  44   Neuronio  45   Neuronio  46   Neuronio  47   Neuronio  48   Neuronio  49   Neuronio  50   Neuronio  51   Neuronio  52   Neuronio  53   Neuronio  54   Neuronio  55   Neuronio  56   Neuronio  57   Neuronio  58   Neuronio  59   Neuronio  60   Neuronio  61   Neuronio  62   Neuronio  63   Neuronio  64   Neuroni

In [ ]:
output_fc3=F.max_pool2d(F.relu(model.conv2(F.max_pool2d(F.relu(model.conv1(model.quant_inp(image))),2))), 2)
output_fc3=output_fc3.reshape(output_fc3.shape[0], -1)
output_fc3=F.relu(model.fc1(output_fc3))
output_fc3=F.relu(model.fc2(output_fc3))
print_layer_output(model, model.fc3, output_fc3, layer_name="Fc3")


Camada: Fc3

ID: Neuronio   0   Neuronio   1   Neuronio   2   Neuronio   3   Neuronio   4   Neuronio   5   Neuronio   6   Neuronio   7   Neuronio   8   Neuronio   9   

0:    -33       -6       -5        9      -64      -21     -123       98      -15        9   
